# Feature Engineering en SQL

A continuación, veremos cómo calcular diferentes variables para el feature engineering utilizando SQL.


In [ ]:
#%pip install jupysql
#%pip install duckdb-engine

In [1]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb://

In [2]:
dataset_path = "/Users/carof/Downloads/DENICOLAY/"
dataset_file = 'competencia_01.csv'


In [3]:
%%sql
create or replace table competencia_01 as
select
    *
from read_csv_auto("{{dataset_path + dataset_file}}")

,Success


### Comparaciones entre dos variables

In [4]:
%%sql
select
    Master_Fvencimiento
    , Visa_Fvencimiento
    , greatest(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_mayor
    , least(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_menor
from competencia_01 limit 10

,Master_Fvencimiento,Visa_Fvencimiento,tc_fvencimiento_mayor,tc_fvencimiento_menor
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,-698,-517,-517,-698
5,-668,-487,-487,-668
6,-637,-456,-456,-637
7,-607,-426,-426,-607
8,-364,-394,-364,-394
9,-334,-364,-334,-364


In [ ]:
#ver como trabajar NAS --> ESTA ABAJO

Lo siguiente es querer operar dos variables, como por ejemplo sumarla. Esto es sencillo


In [5]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,0.0,30874.72,30874.72
5,0.0,51680.48,51680.48
6,0.0,72595.20,72595.20
7,0.0,47478.16,47478.16
8,0.0,29927.43,29927.43
9,0.0,22214.61,22214.61


Pero un DS de a de veras mirará los datos y se encontrará con un campo que es null cuando se lo suma a otro dará null.

In [6]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,63848.46,NaN
5,NaN,NaN,NaN
6,NaN,246.45,NaN
7,NaN,8228.35,NaN
8,NaN,8233.42,NaN
9,NaN,11701.49,NaN


Esto no siempre es deseable y puede ser fácilmente evitable

In [7]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , ifnull(Master_msaldototal, 0) + ifnull(Visa_msaldototal, 0) as tc_saldo_total
from competencia_01 limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,NaN,0.00
1,NaN,NaN,0.00
2,NaN,NaN,0.00
3,NaN,NaN,0.00
4,0.0,30874.72,30874.72
5,0.0,51680.48,51680.48
6,0.0,72595.20,72595.20
7,0.0,47478.16,47478.16
8,0.0,29927.43,29927.43
9,0.0,22214.61,22214.61


### Macros

In [8]:
%%sql
CREATE OR REPLACE MACRO suma_sin_null(a, b) AS ifnull(a, 0) + ifnull(b, 0);


,Success


In [9]:
%%sql
select distinct
    Master_msaldototal
    , Visa_msaldototal
    , suma_sin_null(Master_msaldototal, Visa_msaldototal) as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10


,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,24081.55,24081.55
1,NaN,10056.42,10056.42
2,NaN,27081.97,27081.97
3,NaN,84438.32,84438.32
4,NaN,19881.27,19881.27
5,NaN,36391.02,36391.02
6,NaN,68550.76,68550.76
7,NaN,34065.31,34065.31
8,NaN,11828.42,11828.42
9,NaN,-12308.94,-12308.94


TAREA: Escriba una macro para hacer un ratio de dos variables que sea seguro, donde no solo hay campos con null, también esta el problema de la división por cero. Como es costumbre comparta su solución por este canal. Lea https://duckdb.org/docs/sql/functions/numeric.html para referencias de funciones que puede usar.



In [ ]:
# MI TAREA
#%%sql
#CREATE OR REPLACE MACRO ratio_seguro(a, b) AS ifnull(a, 0) / ifnull(b, 0.1); # PARA NADA RAZONABLE DIVIDIR POR UN NUM ARBITRARIO
# GEMINI SUGIERE MANTENER COMO NULL SEA NAN o SEA 0. PERO QUE HACER CON EL NUMERADOR?

SyntaxError: invalid syntax (299857036.py, line 3)

---

"Claro!" me dirá, mientras lee esto con un mate en la mano, "para cosas fáciles usar SQL alcanza, pero para algo más complicado como crear campos contra el data drifting es difícil".... elija su medicina:

### Window functions

In [ ]:
%%sql
select
    foto_mes
    , numero_de_cliente
    , cliente_antiguedad
    , row_number() over (partition by numero_de_cliente order by foto_mes) as cliente_antiguedad_2 
    # Antiguedad del cliente en meses OBSERVADOS (no distingue intermitencias)
    , percent_rank() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_3 # clientes mas antiguos por mes
    # Para cada mes: Cliente Mas nuevo = 0, Mas antiguo = 1, intermedios: rango marcado como proporcion intermedia
    # Fracción de clientes con menor antigüedad que el cliente actual (en el mes)
    , cume_dist() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_4 # clientes mas antiguos por mes, dif con anterior no me queda clara
    # Para cada mes: Distrib acumulada de clientes menos antiguos respecto de la fila en cuestion
    # # Fracción de clientes con antigüedad menor o igual al cliente actual (en el mes)
    , ntile(4) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_5
    , ntile(10) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_6
    # Pertenencia a cuartiles / deciles: "ranking" dado por pertenencia a particion de menos antiguedad (numeros mas bajos pertenecen a x% menos antiguo versus numeros más altos pertenecen a partición de x% más antiguo)
from competencia_01
order by numero_de_cliente, cliente_antiguedad


,foto_mes,numero_de_cliente,cliente_antiguedad,cliente_antiguedad_2,cliente_antiguedad_3,cliente_antiguedad_4,cliente_antiguedad_5,cliente_antiguedad_6
0,202103,12159854,134,1,0.545491,0.550712,3,6
1,202104,12159854,135,2,0.547454,0.552706,3,6
2,202105,12159854,136,3,0.549647,0.554852,3,6
3,202106,12159854,137,4,0.551510,0.556704,3,6
4,202107,12159854,138,5,0.553128,0.558303,3,6
...,...,...,...,...,...,...,...,...
983056,202108,78249586,1,1,0.000000,0.001585,1,1
983057,202108,78249850,1,1,0.000000,0.001585,1,1
983058,202108,78250419,1,1,0.000000,0.001585,1,1
983059,202108,78253625,1,1,0.000000,0.001585,1,1


Qué paso? use las hermosas funciones analíticas de SQL. Al campo cliente_antiguedad (que no sufre de data drifting, solo esta para dar el ejemplo) para cada período (partition by foto_mes) la ordeno (order by cliente_antiguedad) y luego calculo las métricas de orden que pueden encontrar acá https://duckdb.org/docs/sql/window_functions.html#general-purpose-window-functions.

Seguiremos usando las funciones analíticas de SQL, esta vez para calcular features que utilizan valores del pasado.

Qué pasa si quiero agregar un feature que muestre el valor del periodo anterior?


In [11]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
from competencia_01
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter
0,12160484,202103,65,<NA>
1,12160484,202104,52,65
2,12160484,202105,40,52
3,12160484,202106,33,40
4,12160484,202107,28,33
5,12160484,202108,35,28
6,12160968,202103,60,<NA>
7,12160968,202104,53,60
8,12160968,202105,61,53
9,12160968,202106,58,61


Podemos calcular el delta (diferencia) entre el valor pasado y el presente, para uno o varios meses


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , ctrx_quarter - lag_1_ctrx_quarter as delta_1_ctrx_quarter
  , ctrx_quarter - lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
from competencia_01
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter,delta_1_ctrx_quarter,lag_2_ctrx_quarter
0,14606456,202108,131,115,16,-7
1,14607502,202103,283,<NA>,<NA>,<NA>
2,14607502,202104,278,283,-5,<NA>
3,14607502,202105,250,278,-28,-33
4,14607502,202106,241,250,-9,-37
5,14607502,202107,232,241,-9,-18
6,14607502,202108,255,232,23,14
7,14608983,202103,524,<NA>,<NA>,<NA>
8,14608983,202104,522,524,-2,<NA>
9,14608983,202105,485,522,-37,-39


: 

Si necesitamos ya no solo traer un valor del pasado, sino una secuencia de valores, se puede hacer fácilmente. Por ejemplo, para calcular la media móvil con los últimos 3 meses anteriores:


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
  , lag(ctrx_quarter, 3) over (partition by numero_de_cliente order by foto_mes) as lag_3_ctrx_quarter
  , avg(ctrx_quarter) over (partition by numero_de_cliente
                            order by foto_mes
                            rows between 3 preceding and current row) as avg_3_ctrx_quarter
from competencia_01
order by numero_de_cliente, foto_mes desc
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter,lag_2_ctrx_quarter,lag_3_ctrx_quarter,avg_3_ctrx_quarter
0,12159854,202108,43,45,44,54,46.500000
1,12159854,202107,45,44,54,59,50.500000
2,12159854,202106,44,54,59,64,55.250000
3,12159854,202105,54,59,64,<NA>,59.000000
4,12159854,202104,59,64,<NA>,<NA>,61.500000
5,12159854,202103,64,<NA>,<NA>,<NA>,64.000000
6,12159858,202108,70,68,60,64,65.500000
7,12159858,202107,68,60,64,71,65.750000
8,12159858,202106,60,64,71,65,65.000000
9,12159858,202105,64,71,65,<NA>,66.666667


Si embargo puede resultar incómodo escribir constantemente el over partition sobre todo si se buscan aplicar muchas veces para distintas funciones. Para reducir el código se puede usar la siguiente sintaxis:

In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , avg(ctrx_quarter) over ventana_3 as ctrx_quarter_media_3
  , max(ctrx_quarter) over ventana_3 as ctrx_quarter_max_3
  , min(ctrx_quarter) over ventana_3 as ctrx_quarter_min_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,ctrx_quarter_media_3,ctrx_quarter_max_3,ctrx_quarter_min_3
0,12162107,202103,58,58.000000,58,58
1,12162107,202104,40,49.000000,58,40
2,12162107,202105,32,43.333333,58,32
3,12162107,202106,18,37.000000,58,18
4,12162107,202107,14,26.000000,40,14
5,12162107,202108,14,19.500000,32,14
6,12163274,202103,116,116.000000,116,116
7,12163274,202104,129,122.500000,129,116
8,12163274,202105,125,123.333333,129,116
9,12163274,202106,129,124.750000,129,116


Para saber más que funciones tenemos disponibles, recomiendo ver los siguientes links:

https://duckdb.org/docs/archive/0.8.1/sql/window_functions

https://duckdb.org/docs/archive/0.8.1/sql/aggregates

Un caso más, que ni me voy a molestar en explicar que significa...


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , regr_slope(ctrx_quarter, cliente_antiguedad) over ventana_3 as ctrx_quarter_slope_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,ctrx_quarter_slope_3
0,12173792,202103,175,NaN
1,12173792,202104,190,15.0
2,12173792,202105,213,19.0
3,12173792,202106,237,20.9
4,12173792,202107,250,20.4
5,12173792,202108,285,22.9
6,12182374,202103,144,NaN
7,12182374,202104,155,11.0
8,12182374,202105,162,9.0
9,12182374,202106,154,3.7


 regr_slope(y, x) toma primero la variable dependiente $y$ y luego la independiente $x$. 
 Al pasar regr_slope(var, cliente_antiguedad) estás calculando la tasa de cambio de la variable a lo largo del tiempo de antigüedad, lo cual es matemáticamente correcto.

... Alguno dirá "tenemos que escribir todo esto a mano? Son muchas variables!". 

Bueno, no, use los conocimientos de programación para que la computadora trabaje para usted. Si tenemos una lista de campos.....

In [ ]:
campos = ['active_quarter', 'cliente_vip', 'internet', 'cliente_edad', 'cliente_antiguedad', 'mrentabilidad']


Podemos hacer un script muy sencillo que nos genere el texto que hay que poner en una query para generar esas variables


In [ ]:
nuevos_features = ""
for campo in campos:
  nuevos_features += f"\n, regr_slope({campo}, cliente_antiguedad) over ventana_3 as ctrx_{campo}_slope_3"
print(nuevos_features)



, regr_slope(active_quarter, cliente_antiguedad) over ventana_3 as ctrx_active_quarter_slope_3
, regr_slope(cliente_vip, cliente_antiguedad) over ventana_3 as ctrx_cliente_vip_slope_3
, regr_slope(internet, cliente_antiguedad) over ventana_3 as ctrx_internet_slope_3
, regr_slope(cliente_edad, cliente_antiguedad) over ventana_3 as ctrx_cliente_edad_slope_3
, regr_slope(cliente_antiguedad, cliente_antiguedad) over ventana_3 as ctrx_cliente_antiguedad_slope_3
, regr_slope(mrentabilidad, cliente_antiguedad) over ventana_3 as ctrx_mrentabilidad_slope_3


Con la salida de esa celda, arme la query agregando las nuevas líneas y ejecutela.


In [ ]:
%%sql
select
  numero_de_cliente
 , foto_mes
 , ctrx_quarter
 , regr_slope(active_quarter, cliente_antiguedad) over ventana_3 as ctrx_active_quarter_slope_3
 , regr_slope(cliente_vip, cliente_antiguedad) over ventana_3 as ctrx_cliente_vip_slope_3
 , regr_slope(internet, cliente_antiguedad) over ventana_3 as ctrx_internet_slope_3
 , regr_slope(cliente_edad, cliente_antiguedad) over ventana_3 as ctrx_cliente_edad_slope_3
 , regr_slope(cliente_antiguedad, cliente_antiguedad) over ventana_3 as ctrx_cliente_antiguedad_slope_3
 , regr_slope(mrentabilidad, cliente_antiguedad) over ventana_3 as ctrx_mrentabilidad_slope_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10

Lo que acabamos de hacer de manera muy simple es como "funcionan" sistemas como **dbt** que están tan de moda en el mundo de los datos.

Una última reflexión: la creación de nuevas features es un proceso computacionalmente rápido pero intenso. Si ejecutó lo anterior pudo haber visto que en poco minutos tenía sus nuevas variables. Pero, también pudo haberle fallado por temas de recursos. Miles de variables necesitan los recursos adecuados. Use la nube, una máquina grande, al menos que sepa bien como optimizar las queries.

Y a no olvidarse guardar las nueva tabla

In [ ]:
#%%sql
#COPY competencia_01 TO '{dataset_path}competencia_01_fe.csv' (FORMAT CSV, HEADER TRUE);

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


PARA GUARDAR, SEGUN GEMINI

¿Cómo guardarlas correctamente?
Tienes dos alternativas estándar en DuckDB:

**Opción 1: Exportar directamente la consulta al CSV*** (la más limpia y sin ocupar espacio extra)
Puedes envolver el SELECT (sin el LIMIT 10) directamente en el comando COPY:

In [ ]:
%%sql
# EJEMPLO
COPY (
    select
        numero_de_cliente
        , foto_mes
        , ctrx_quarter
        , regr_slope(active_quarter, cliente_antiguedad) over ventana_3 as ctrx_active_quarter_slope_3
        , regr_slope(cliente_vip, cliente_antiguedad) over ventana_3 as ctrx_cliente_vip_slope_3
        , regr_slope(internet, cliente_antiguedad) over ventana_3 as ctrx_internet_slope_3
        , regr_slope(cliente_edad, cliente_antiguedad) over ventana_3 as ctrx_cliente_edad_slope_3
        , regr_slope(cliente_antiguedad, cliente_antiguedad) over ventana_3 as ctrx_cliente_antiguedad_slope_3
        , regr_slope(mrentabilidad, cliente_antiguedad) over ventana_3 as ctrx_mrentabilidad_slope_3
    from competencia_01
    window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
) TO '{dataset_path}competencia_01_fe.csv' (FORMAT CSV, HEADER TRUE);

(Si quieres mantener todas las columnas originales además de las pendientes, agrega competencia_01.* al inicio del SELECT).

**Opción 2: Crear una nueva tabla persistente y luego exportarla**
Si prefieres tener la tabla disponible en la sesión de DuckDB para seguir trabajando:

In [ ]:
%%sql
# SEGUNDO EJEMPLO
# -- 1. Crear una nueva tabla con las features añadidas
CREATE TABLE competencia_01_fe AS
select
    competencia_01.*
    , regr_slope(active_quarter, cliente_antiguedad) over ventana_3 as ctrx_active_quarter_slope_3
    , regr_slope(cliente_vip, cliente_antiguedad) over ventana_3 as ctrx_cliente_vip_slope_3
    , regr_slope(internet, cliente_antiguedad) over ventana_3 as ctrx_internet_slope_3
    , regr_slope(cliente_edad, cliente_antiguedad) over ventana_3 as ctrx_cliente_edad_slope_3
    , regr_slope(cliente_antiguedad, cliente_antiguedad) over ventana_3 as ctrx_cliente_antiguedad_slope_3
    , regr_slope(mrentabilidad, cliente_antiguedad) over ventana_3 as ctrx_mrentabilidad_slope_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row);

#-- 2. Exportar la nueva tabla
COPY competencia_01_fe TO '{dataset_path}competencia_01_fe.csv' (FORMAT CSV, HEADER TRUE);

# **TAREA: CREO NUEVAS FEATURES Y LAS GUARDO**

Armen variables, FE intra mes, FE historico, lo van a tener que tener listo para el clustering. 
Doy por descontaddo que van a correr un RF sobre el dataset ampliado y analizar las variables historicas (o decirles a claudio que las haga)

Yo obvio que lo hago con ayuda de Gemini

### Variables intra mes

1. Features Intra-Mes (Foto Actual)

Estas variables capturan ratios de intensidad, bancarización, riesgo y uso del crédito dentro del mismo mes $t$:

Agregaciones Totales de Tarjetas de Crédito:

mconsumototal_tc = mtarjeta_visa_consumo + mtarjeta_master_consumocconsumos_total_tc = ctarjeta_visa_transacciones + ctarjeta_master_transaccionesmsaldototal_tc = Visa_msaldototal + Master_msaldototalmlimitecompra_total = Visa_mlimitecompra + Master_mlimitecomprampagominimo_total = Visa_mpagominimo + Master_mpagominimompagado_total = Visa_mpagado + Master_mpagado

Ratios de Utilización y Apalancamiento:

ratio_utilizacion_limite = msaldototal_tc / (mlimitecompra_total + 0.01) (señales de saturación crediticia o desuso)ratio_pago_vs_saldo = mpagado_total / (msaldototal_tc + 0.01)ratio_pago_minimo = mpagado_total / (mpagominimo_total + 0.01) (si paga exactamente el mínimo o menos, hay estrés financiero)

Patrimonio Líquido y Pasivos:

saldo_liquido_total = mcuentas_saldo + mplazo_fijo_pesos + minversion1_pesos + minversion2deuda_prestamos_total = mprestamos_personales + mprestamos_prendarios + mprestamos_hipotecariosratio_liquidez_vs_deuda = saldo_liquido_total / (deuda_prestamos_total + 0.01)

Intensidad Digital vs. Físico:

trx_digitales = chomebanking_transacciones + cmobile_app_trxtrx_fisicas = ccajas_transacciones + catm_trx + ccallcenter_transaccionesratio_canal_digital = trx_digitales / (trx_digitales + trx_fisicas + 0.01)

Ticket Promedio Transaccional:

ticket_promedio_visa = mtarjeta_visa_consumo / (ctarjeta_visa_transacciones + 0.01)ticket_promedio_master = mtarjeta_master_consumo / (ctarjeta_master_transacciones + 0.01)ticket_promedio_debito = mautoservicio / (ctarjeta_debito_transacciones + 0.01)

Diferencia Temporal en Tarjetas:

Antigüedad de la tarjeta respecto a la foto actual: antiguedad_dias_visa = foto_mes - Visa_fechaalta (convirtiendo a días o meses).Días desde el último vencimiento / inicio de mora.

In [4]:
%%sql
CREATE OR REPLACE TABLE competencia_01_fe AS
WITH base_agregaciones AS (
    SELECT
        *,

        -- 1. Agregaciones totales de Tarjetas de Crédito (Visa + Master)
        (COALESCE(mtarjeta_visa_consumo, 0) + COALESCE(mtarjeta_master_consumo, 0)) AS mconsumototal_tc,
        (COALESCE(ctarjeta_visa_transacciones, 0) + COALESCE(ctarjeta_master_transacciones, 0)) AS cconsumos_total_tc,
        (COALESCE(Visa_msaldototal, 0) + COALESCE(Master_msaldototal, 0)) AS msaldototal_tc,
        (COALESCE(Visa_mlimitecompra, 0) + COALESCE(Master_mlimitecompra, 0)) AS mlimitecompra_total,
        (COALESCE(Visa_mpagominimo, 0) + COALESCE(Master_mpagominimo, 0)) AS mpagominimo_total,
        (COALESCE(Visa_mpagado, 0) + COALESCE(Master_mpagado, 0)) AS mpagado_total,

        -- 2. Patrimonio Líquido y Deuda en Préstamos
        (COALESCE(mcuentas_saldo, 0) + COALESCE(mplazo_fijo_pesos, 0) + 
         COALESCE(minversion1_pesos, 0) + COALESCE(minversion2, 0)) AS saldo_liquido_total,
        (COALESCE(mprestamos_personales, 0) + COALESCE(mprestamos_prendarios, 0) + 
         COALESCE(mprestamos_hipotecarios, 0)) AS deuda_prestamos_total,

        -- 3. Canales Digitales vs Físicos
        (COALESCE(chomebanking_transacciones, 0) + COALESCE(cmobile_app_trx, 0)) AS trx_digitales,
        (COALESCE(ccajas_transacciones, 0) + COALESCE(catm_trx, 0) + COALESCE(ccallcenter_transacciones, 0)) AS trx_fisicas,

        -- 4. Normalización de fechas para cálculo de antigüedad
        strptime(CAST(foto_mes AS VARCHAR) || '01', '%Y%m%d')::DATE AS fecha_foto,
        CASE 
            WHEN Visa_fechaalta IS NOT NULL AND Visa_fechaalta > 0 
            THEN try_strptime(CAST(Visa_fechaalta AS VARCHAR), '%Y%m%d')::DATE 
            ELSE NULL 
        END AS visa_fechaalta_date,
        CASE 
            WHEN Master_fechaalta IS NOT NULL AND Master_fechaalta > 0 
            THEN try_strptime(CAST(Master_fechaalta AS VARCHAR), '%Y%m%d')::DATE 
            ELSE NULL 
        END AS master_fechaalta_date

    FROM competencia_01
)
SELECT
    * EXCLUDE (fecha_foto, visa_fechaalta_date, master_fechaalta_date),

    -- Ratios de Tarjetas de Crédito
    ROUND(msaldototal_tc / NULLIF(mlimitecompra_total, 0), 4) AS ratio_utilizacion_limite,
    ROUND(mpagado_total / NULLIF(msaldototal_tc, 0), 4) AS ratio_pago_vs_saldo,
    ROUND(mpagado_total / NULLIF(mpagominimo_total, 0), 4) AS ratio_pago_minimo,

    -- Solvencia y Liquidez
    ROUND(saldo_liquido_total / NULLIF(deuda_prestamos_total, 0), 4) AS ratio_liquidez_vs_deuda,

    -- Intensidad Digital
    ROUND(trx_digitales / NULLIF(trx_digitales + trx_fisicas, 0), 4) AS ratio_canal_digital,

    -- Tickets Promedio
    ROUND(mtarjeta_visa_consumo / NULLIF(ctarjeta_visa_transacciones, 0), 2) AS ticket_promedio_visa,
    ROUND(mtarjeta_master_consumo / NULLIF(ctarjeta_master_transacciones, 0), 2) AS ticket_promedio_master,
    ROUND(mautoservicio / NULLIF(ctarjeta_debito_transacciones, 0), 2) AS ticket_promedio_debito,

    -- Antigüedades en meses de tarjetas
    DATE_DIFF('month', visa_fechaalta_date, fecha_foto) AS antiguedad_meses_visa,
    DATE_DIFF('month', master_fechaalta_date, fecha_foto) AS antiguedad_meses_master

FROM base_agregaciones;

,Success


### Features historicas

2. Features Históricas (Ventana Temporal)El churn casi nunca es abrupto; suele haber entre 2 y 6 meses de abandono progresivo. Las siguientes operaciones deben aplicarse agrupando por numero_de_cliente y ordenando por foto_mes.Variables clave a trackear en el tiempo:Transaccionalidad: ctrx_quarter, cproductos, chomebanking_transacciones, cmobile_app_trx, cconsumos_total_tc.Montos: mcuentas_saldo, mpayroll, mrentabilidad, mconsumototal_tc.Transformaciones temporales recomendadas:Deltas y Derivadas (Diferencias Absolutas):$\Delta_1 = X_t - X_{t-1}$ (pérdida mensual directa)$\Delta_2 = X_t - X_{t-2}$$\Delta_3 = X_t - X_{t-3}$$\Delta_6 = X_t - X_{t-6}$Ratios de Variación (Cambios Relativos):$Ratio_1 = \frac{X_t + 1}{X_{t-1} + 1}$$Ratio_3 = \frac{X_t + 1}{X_{t-3} + 1}$(Si el ratio de transacciones o saldo cae fuertemente hacia 0, la probabilidad de churn se dispara).Medias Móviles y Tendencias (Rolling Windows de 3 y 6 meses):mean_3m, mean_6mratio_actual_vs_media3m = X_t / (mean_3m + 0.01)tendencia_lineal: pendiente (slope) de los últimos 3 o 6 meses para variables como saldo, rentabilidad o transacciones.Volatilidad y Estabilidad:std_3m, std_6m (clientes con caída a desviación cero en transacciones indican inactividad total).Mínimos, Máximos y Distancia a Picos:max_6m, min_6mdistancia_al_pico = X_t / (max_6m + 0.01) (indica qué tan lejos está el cliente de su nivel máximo de transacciones o saldo de los últimos 6 meses).3. Señales Directas de Abandono (Flags y Conteos de Eventos)Cesación de Sueldo (mpayroll / cpayroll_trx):Flag si mpayroll_t == 0 y mpayroll_{t-1} > 0 (cambio de banco para cobro de haberes).Vaciado de Cuentas:Flag si mcuentas_saldo_t <= 0 habiendo tenido saldo positivo en $t-1$ o $t-2$.Inactividad de Tarjetas:Meses consecutivos sin consumos en Visa y Master: meses_sin_consumo_tc.Fuga de Débitos Automáticos:$\Delta(ccuenta\_debitos\_automaticos + ctarjeta\_visa\_debitos\_automaticos + ctarjeta\_master\_debitos\_automaticos)$ a 3 meses.Mora y Cobranzas:Flag de cambio en Master_status o Visa_status.Master_delinquency_t - Master_delinquency_{t-1} (incremento de días de atraso).

In [5]:
%%sql
CREATE OR REPLACE TABLE competencia_01_hist AS
WITH base_lags AS (
    SELECT
        *,

        -- =========================================================================
        -- 1. LAGS TEMPORALES (t-1, t-2, t-3, t-6)
        -- =========================================================================
        -- Transaccionales
        LAG(ctrx_quarter, 1) OVER w AS ctrx_quarter_lag1,
        LAG(ctrx_quarter, 2) OVER w AS ctrx_quarter_lag2,
        LAG(ctrx_quarter, 3) OVER w AS ctrx_quarter_lag3,
        LAG(ctrx_quarter, 6) OVER w AS ctrx_quarter_lag6,

        LAG(cproductos, 1) OVER w AS cproductos_lag1,
        LAG(cproductos, 2) OVER w AS cproductos_lag2,
        LAG(cproductos, 3) OVER w AS cproductos_lag3,

        LAG(chomebanking_transacciones, 1) OVER w AS chb_trx_lag1,
        LAG(cmobile_app_trx, 1)            OVER w AS cmobile_trx_lag1,
        LAG(cconsumos_total_tc, 1)         OVER w AS cconsumos_tc_lag1,
        LAG(cconsumos_total_tc, 3)         OVER w AS cconsumos_tc_lag3,

        -- Monetarios
        LAG(mcuentas_saldo, 1) OVER w AS mcuentas_saldo_lag1,
        LAG(mcuentas_saldo, 2) OVER w AS mcuentas_saldo_lag2,
        LAG(mcuentas_saldo, 3) OVER w AS mcuentas_saldo_lag3,
        LAG(mcuentas_saldo, 6) OVER w AS mcuentas_saldo_lag6,

        LAG(mpayroll, 1) OVER w AS mpayroll_lag1,
        LAG(mpayroll, 2) OVER w AS mpayroll_lag2,

        LAG(mrentabilidad, 1) OVER w AS mrentabilidad_lag1,
        LAG(mrentabilidad, 3) OVER w AS mrentabilidad_lag3,

        LAG(mconsumototal_tc, 1) OVER w AS mconsumototal_tc_lag1,
        LAG(mconsumototal_tc, 3) OVER w AS mconsumototal_tc_lag3,

        -- =========================================================================
        -- 2. ROLLING WINDOWS (Medias, Desvíos, Mínimos y Máximos)
        -- =========================================================================
        -- Ventanas de 3 meses (mes actual + 2 precedentes)
        AVG(ctrx_quarter)       OVER w3 AS ctrx_quarter_avg3m,
        STDDEV(ctrx_quarter)    OVER w3 AS ctrx_quarter_std3m,
        AVG(mcuentas_saldo)     OVER w3 AS mcuentas_saldo_avg3m,
        STDDEV(mcuentas_saldo)  OVER w3 AS mcuentas_saldo_std3m,
        AVG(mrentabilidad)      OVER w3 AS mrentabilidad_avg3m,
        AVG(mconsumototal_tc)   OVER w3 AS mconsumototal_tc_avg3m,

        -- Ventanas de 6 meses (mes actual + 5 precedentes) NO SE QUE TAN UTIL VA A SER ESTO ME HACE RUIDO, DEPENDE DE COMO ENTRENEMOS
        AVG(ctrx_quarter)       OVER w6 AS ctrx_quarter_avg6m,
        MAX(ctrx_quarter)       OVER w6 AS ctrx_quarter_max6m,
        MIN(ctrx_quarter)       OVER w6 AS ctrx_quarter_min6m,
        AVG(mcuentas_saldo)     OVER w6 AS mcuentas_saldo_avg6m,
        MAX(mcuentas_saldo)     OVER w6 AS mcuentas_saldo_max6m,
        MIN(mcuentas_saldo)     OVER w6 AS mcuentas_saldo_min6m

    FROM competencia_01_fe
    WINDOW 
        w  AS (PARTITION BY numero_de_cliente ORDER BY foto_mes),
        w3 AS (PARTITION BY numero_de_cliente ORDER BY foto_mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),
        w6 AS (PARTITION BY numero_de_cliente ORDER BY foto_mes ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)
)
SELECT
    *,

    -- =========================================================================
    -- 3. DELTAS ABSOLUTOS (Pérdida/Ganancia directa)
    -- =========================================================================
    (ctrx_quarter - ctrx_quarter_lag1) AS ctrx_quarter_delta1,
    (ctrx_quarter - ctrx_quarter_lag2) AS ctrx_quarter_delta2,
    (ctrx_quarter - ctrx_quarter_lag3) AS ctrx_quarter_delta3,
    (ctrx_quarter - ctrx_quarter_lag6) AS ctrx_quarter_delta6,

    (cproductos - cproductos_lag1) AS cproductos_delta1,
    (cproductos - cproductos_lag3) AS cproductos_delta3,

    (mcuentas_saldo - mcuentas_saldo_lag1) AS mcuentas_saldo_delta1,
    (mcuentas_saldo - mcuentas_saldo_lag2) AS mcuentas_saldo_delta2,
    (mcuentas_saldo - mcuentas_saldo_lag3) AS mcuentas_saldo_delta3,
    (mcuentas_saldo - mcuentas_saldo_lag6) AS mcuentas_saldo_delta6,

    (mrentabilidad - mrentabilidad_lag1) AS mrentabilidad_delta1,
    (mrentabilidad - mrentabilidad_lag3) AS mrentabilidad_delta3,

    (mconsumototal_tc - mconsumototal_tc_lag1) AS mconsumototal_tc_delta1,
    (mconsumototal_tc - mconsumototal_tc_lag3) AS mconsumototal_tc_delta3,

    -- =========================================================================
    -- 4. RATIOS RELATIVOS DE VARIACIÓN
    -- =========================================================================
    ROUND((ctrx_quarter + 1.0) / NULLIF(ctrx_quarter_lag1 + 1.0, 0), 4) AS ctrx_quarter_ratio1, # ACA ESTA LA SOLUCION A LA MACRO DE ARRIBA.. SUMAR POR 1 DE LOS DOS LADOS DE LA DIVISION
    ROUND((ctrx_quarter + 1.0) / NULLIF(ctrx_quarter_lag3 + 1.0, 0), 4) AS ctrx_quarter_ratio3,

    ROUND((cconsumos_total_tc + 1.0) / NULLIF(cconsumos_tc_lag1 + 1.0, 0), 4) AS cconsumos_tc_ratio1,
    ROUND((cconsumos_total_tc + 1.0) / NULLIF(cconsumos_tc_lag3 + 1.0, 0), 4) AS cconsumos_tc_ratio3,

    -- Ratios contra promedios móviles
    ROUND(ctrx_quarter / NULLIF(ctrx_quarter_avg3m, 0), 4)   AS ctrx_quarter_ratio_avg3m,
    ROUND(mcuentas_saldo / NULLIF(mcuentas_saldo_avg3m, 0), 4) AS mcuentas_saldo_ratio_avg3m,

    -- Distancia al pico (Drawdown de actividad y saldos respecto al máximo histórico a 6m) # IDEM, NO TIENEN SENTIDO PARA MI
    ROUND(ctrx_quarter / NULLIF(ctrx_quarter_max6m, 0), 4)   AS ctrx_distancia_pico6m,
    ROUND(mcuentas_saldo / NULLIF(mcuentas_saldo_max6m, 0), 4) AS mcuentas_saldo_distancia_pico6m,

    -- =========================================================================
    -- 5. SEÑALES DIRECTAS DE ALERTA DE CHURN (Flags Binarios)
    -- =========================================================================
    -- Dejó de cobrar sueldo (tenía payroll y cayó a 0)
    CASE 
        WHEN COALESCE(mpayroll, 0) = 0 AND COALESCE(mpayroll_lag1, 0) > 0 THEN 1 
        ELSE 0 
    END AS flag_cesacion_payroll,

    -- Vaciado abrupto de cuentas corrientes / cajas de ahorro
    CASE 
        WHEN COALESCE(mcuentas_saldo, 0) <= 0 AND COALESCE(mcuentas_saldo_lag1, 0) > 0 THEN 1 
        ELSE 0 
    END AS flag_vaciado_cuentas,

    -- Abandono total de tarjetas de crédito
    CASE 
        WHEN COALESCE(cconsumos_total_tc, 0) = 0 AND COALESCE(cconsumos_tc_lag1, 0) > 0 THEN 1 
        ELSE 0 
    END AS flag_abandono_consumo_tc

FROM base_lags;

,Success


### Guardado

In [21]:
output_path = f"{dataset_path}competencia_01_fe.parquet"

In [16]:
output_path

'/Users/carof/Downloads/DENICOLAY/competencia_01_fe.parquet'

In [24]:
%%sql
COPY competencia_01_hist TO '{{output_path}}' (FORMAT PARQUET, COMPRESSION ZSTD);

,Success


In [25]:
output_path = f"{dataset_path}competencia_01_fe.csv"

In [26]:
%%sql
COPY competencia_01_hist TO "{{output_path}}" (HEADER, DELIMITER ',');

,Success
